# 🌱 Atmosphäre-Analyse
## Layer 2 – Oberfläche & Biosphärennahe Kontaktzone

| Layer | Name | Status |
|-------|------|--------|
| 0 | Externe kosmische Einflüsse | ✅ `layer0_state.json` |
| 1 | Geophysikalischer Grundzustand | ✅ `layer1_state.json` |
| **2** | **Oberfläche & Biosphärennahe Kontaktzone** | **← dieser Layer** |
| 3 | Atmosphäre / Wetter / Gewitter | ⬜ |
| 4 | Ionosphäre | ⬜ |
| 5 | Global Electric Circuit | ⬜ |
| 6 | Resonanz- und Musterfeld | ⬜ |
| 7 | Interpretation / Systemzustand | ⬜ |

> **Kernidee:** Hier beginnt die energetische und materielle Kopplung mit der Atmosphäre. Layer 2 moduliert Konvektion, Gewitterbildung und elektrische Austauschprozesse.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import datetime, json, math, requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# --- Projektpfade (CWD-unabhaengig, ohne pip install) ---
import sys, pathlib
_root = pathlib.Path.cwd().resolve()
while not (_root / '.project-root').exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root / 'src'))
from atmosphere.paths import layer_state

print(f'Pakete geladen')
print(f'Analysedatum: {datetime.date.today()}')

# Vorherige Layer laden
context = {}
for layer_n in [0, 1]:
    try:
        with open(layer_state(layer_n), encoding='utf-8') as f:
            context[layer_n] = json.load(f)
        print(f'  Layer {layer_n}: {context[layer_n]["level"].upper():8}  Score={context[layer_n]["score"]}  Driver/Name: {context[layer_n].get("dominant_driver", context[layer_n]["name"])}')
    except FileNotFoundError:
        print(f'  Layer {layer_n}: nicht gefunden')
        context[layer_n] = None

layer0 = context.get(0)
layer1 = context.get(1)

---
## 1. Systemstruktur: Die 7 Elemente von Layer 2

In [ ]:
elements = [
    {'name': 'Land &<br>Bodentypen',          'x': 0.50, 'y': 0.92, 'color': '#85643A', 'sz': 54},
    {'name': 'Ozeane &<br>SST',               'x': 0.20, 'y': 0.76, 'color': '#185FA5', 'sz': 54},
    {'name': 'Feuchtigkeit<br>& Niederschlag', 'x': 0.80, 'y': 0.76, 'color': '#378ADD', 'sz': 54},
    {'name': 'Temperatur-<br>verteilung',      'x': 0.12, 'y': 0.53, 'color': '#E85D24', 'sz': 52},
    {'name': 'Vegetation &<br>Biosphäre',      'x': 0.50, 'y': 0.57, 'color': '#639922', 'sz': 54},
    {'name': 'Lokale<br>E-Felder',             'x': 0.88, 'y': 0.53, 'color': '#7F77DD', 'sz': 52},
    {'name': 'Boden-Atm.<br>Interaktion',      'x': 0.50, 'y': 0.35, 'color': '#888780', 'sz': 54},
]
core = {'x': 0.50, 'y': 0.14}

fig = go.Figure()
for el in elements:
    dx = core['x'] - el['x']; dy = core['y'] - el['y']
    dist = math.sqrt(dx**2 + dy**2)
    t = 0.05 / dist
    fig.add_trace(go.Scatter(
        x=[el['x'], core['x'] - dx*t], y=[el['y'], core['y'] - dy*t],
        mode='lines', line=dict(color=el['color'], width=1.6), opacity=0.4,
        showlegend=False, hoverinfo='skip'
    ))

fig.add_trace(go.Scatter(
    x=[core['x']], y=[core['y']], mode='markers+text',
    marker=dict(size=82, color='#3B6D11', opacity=0.90, line=dict(color='#C0DD97', width=2)),
    text=['🌱 Kontaktzone<br>Oberfläche'], textposition='middle center',
    textfont=dict(size=10, color='white'), showlegend=False, hoverinfo='skip'
))
for el in elements:
    fig.add_trace(go.Scatter(
        x=[el['x']], y=[el['y']], mode='markers+text',
        marker=dict(size=el['sz'], color=el['color'], opacity=0.88, line=dict(color='white', width=1.8)),
        text=[el['name']], textposition='middle center',
        textfont=dict(size=9.5, color='white'), showlegend=False,
        hovertemplate=el['name'].replace('<br>',' ') + '<extra></extra>'
    ))

for txt, px, py, col in [
    ('🌊  Hydrosphäre & Thermodynamik', 0.50, 1.00, '#0C447C'),
    ('🌿  Biosphäre & Oberfläche',      0.50, 0.64, '#3B6D11'),
    ('⚡  Elektrische Kopplung',         0.50, 0.26, '#534AB7'),
]:
    fig.add_annotation(x=px, y=py, text=txt, showarrow=False,
                       xref='paper', yref='paper', font=dict(size=11, color=col))

# Layer-Kontext
if layer1:
    ctx = f'L0: {context[0]["level"].upper() if layer0 else "–"}  |  L1: {layer1["level"].upper()}  |  Seismik-Flag: {layer1.get("flags",{}).get("elevated_seismicity","–")}'
    fig.add_annotation(x=0.5, y=0.02, text=ctx, showarrow=False,
                       xref='paper', yref='paper', font=dict(size=10, color='#888780'))

fig.update_layout(
    title=dict(text='Layer 2 – Oberfläche & Kontaktzone: Systemstruktur', font=dict(size=16)),
    xaxis=dict(showgrid=False, zeroline=False, visible=False, range=[-0.05, 1.05]),
    yaxis=dict(showgrid=False, zeroline=False, visible=False, range=[0.02, 1.05]),
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    height=540, margin=dict(l=20, r=20, t=55, b=20)
)
fig.show()

---
## 2. Echtdaten abrufen

In [ ]:
# ============================================================
# ECHTDATEN – VIER QUELLEN
# 1) Open-Meteo  – Globale Wetter/Oberflächen-Indikatoren
# 2) NOAA ERSST  – Sea Surface Temperature Anomalie
# 3) NASA FIRMS  – Aktive Feuer / Vegetation-Stress
# 4) NOAA Tide   – Ozean-Pegelstand (Proxy Wärmeausdehnung)
# ============================================================

raw = {}

# --- 1) Open-Meteo: Oberflächenparameter für repräsentative Punkte ---
# Stichprobenpunkte aus der zentralen Single Source (== Layer 3 == Meso)
from atmosphere.config.domain import OM_POINTS
raw['surface'] = []
for pt in OM_POINTS:
    try:
        url = (
            f'https://api.open-meteo.com/v1/forecast'
            f'?latitude={pt["lat"]}&longitude={pt["lon"]}'
            f'&current=temperature_2m,relative_humidity_2m,'
            f'precipitation,surface_pressure,wind_speed_10m,'
            f'soil_moisture_0_to_1cm,et0_fao_evapotranspiration'
            f'&daily=temperature_2m_max,temperature_2m_min,'
            f'precipitation_sum,et0_fao_evapotranspiration'
            f'&forecast_days=1&timezone=UTC'
        )
        r = requests.get(url, timeout=15); r.raise_for_status()
        data = r.json()
        data['_point'] = pt['name']
        raw['surface'].append(data)
        cur = data.get('current', {})
        print(f'  ✅ Open-Meteo {pt["name"]:<18} T={cur.get("temperature_2m","–")}°C  RH={cur.get("relative_humidity_2m","–")}%')
    except Exception as e:
        print(f'  ❌ Open-Meteo {pt["name"]:<18} {str(e)[:60]}')

# --- 2) NOAA ERDDAP: SST-Anomalie (globaler Monatsmittel-Index) ---
try:
    url = ('https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/'
           'global/time-series/globe/ocean/1/1/1880-2024.json')
    r = requests.get(url, timeout=15); r.raise_for_status()
    raw['sst_anomaly'] = r.json()
    vals = raw['sst_anomaly'].get('data', {})
    last_key = sorted(vals.keys())[-1]
    parsed_v = vals[last_key]
    parsed_v = parsed_v['anomaly'] if isinstance(parsed_v, dict) else parsed_v
    print(f'  ✅ NOAA SST-Anomalie  {len(vals)} Monate, aktuell ({last_key}): {float(parsed_v):+.3f} degC')
except Exception as e:
    print(f'  ❌ NOAA SST-Anomalie  {str(e)[:70]}')
    raw['sst_anomaly'] = None

# --- 3) NOAA Geomag-Aktivität (kompaktes JSON, kein Parse-Problem) ---
try:
    url = 'https://services.swpc.noaa.gov/json/planetary_k_index_1m.json'
    r = requests.get(url, timeout=10); r.raise_for_status()
    raw['aurora'] = r.json()
    print(f'  ✅ NOAA Geomag (Kp)   {len(raw["aurora"]):>5} Einträge')
except Exception as e:
    print(f'  ❌ NOAA Geomag        {str(e)[:70]}')
    raw['aurora'] = None

# --- 4) ENSO: CPC Weekly Nino3.4 (primary) + ONI (secondary) ---
# Primary: NOAA CPC weekly SST – Nino3.4 SSTA in fixed-width column
raw['enso_weekly_raw'] = None
raw['enso_oni_raw']    = None

try:
    url = 'https://www.cpc.ncep.noaa.gov/data/indices/wksst9120.for'
    r = requests.get(url, timeout=15); r.raise_for_status()
    raw['enso_weekly_raw'] = r.text
    n = len([l for l in r.text.splitlines() if len(l) > 30])
    print(f'  ✅ NOAA CPC weekly SST {n:>5} Zeilen')
except Exception as e:
    print(f'  ↩  CPC weekly          {str(e)[:70]}')

try:
    url = 'https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt'
    r = requests.get(url, timeout=15); r.raise_for_status()
    raw['enso_oni_raw'] = r.text
    n = len([l for l in r.text.splitlines() if l.strip() and not l.startswith('SEAS')])
    print(f'  ✅ NOAA CPC ONI        {n:>5} Einträge')
except Exception as e:
    print(f'  ↩  CPC ONI             {str(e)[:70]}')
    raw['enso_oni_raw'] = None

print(f'\n📥 Datenabruf abgeschlossen: {len(raw["surface"])}/{len(OM_POINTS)} Messpunkte')




In [ ]:
import re
# ============================================================
# DATEN AUFBEREITEN
# ============================================================

# --- Oberflächenparameter aus Open-Meteo ---
surface_rows = []
for d in raw['surface']:
    cur = d.get('current', {})
    daily = d.get('daily', {})
    surface_rows.append({
        'point':       d['_point'],
        'temp_2m':     cur.get('temperature_2m'),
        'rh':          cur.get('relative_humidity_2m'),
        'precip':      cur.get('precipitation'),
        'pressure':    cur.get('surface_pressure'),
        'wind':        cur.get('wind_speed_10m'),
        'soil_moist':  cur.get('soil_moisture_0_to_1cm'),
        'et0':         cur.get('et0_fao_evapotranspiration'),
        'temp_max':    daily.get('temperature_2m_max', [None])[0],
        'temp_min':    daily.get('temperature_2m_min', [None])[0],
        'precip_day':  daily.get('precipitation_sum', [None])[0],
    })
df_surf = pd.DataFrame(surface_rows)
for col in df_surf.columns[1:]:
    df_surf[col] = pd.to_numeric(df_surf[col], errors='coerce')

print('Oberflächendaten:')
print(df_surf[['point','temp_2m','rh','precip','wind','soil_moist']].to_string(index=False))

# --- SST-Anomalie ---
sst_anomaly_now = None
sst_source = 'missing'
if raw['sst_anomaly']:
    vals = raw['sst_anomaly'].get('data', {})
    if vals:
        last_key = sorted(vals.keys())[-1]
        try:
            raw_v = vals[last_key]
            sst_anomaly_now = float(raw_v['anomaly'] if isinstance(raw_v, dict) else raw_v)
            sst_source = 'primary'
            print(f'\nSST-Anomalie: {sst_anomaly_now:+.3f} degC ({last_key})')
        except: pass

# --- ENSO: Weekly Nino3.4 + ONI ---
enso_now        = None   # weekly Nino3.4 anomaly
enso_oni        = None   # 3-month ONI
enso_source     = 'missing'
enso_phase_obs  = 'unknown'
enso_phase_fcst = 'not_used'

# Weekly Nino3.4 aus CPC fixed-width Format
# Format alt: '02SEP1981  20.6-0.1  24.8-0.1  26.5-0.2  28.3-0.3'
# Format neu: '28APR2026  27.43  0.28  27.81  0.14  28.01 -0.23  29.12  0.18'
# Regex extrahiert alle Dezimalzahlen – beide Formate:
# nach dem Datum: N12_SST[0] N12_SSTA[1] N3_SST[2] N3_SSTA[3] N34_SST[4] N34_SSTA[5]
if raw.get('enso_weekly_raw'):
    try:
        data_lines = [l.strip() for l in raw['enso_weekly_raw'].splitlines()
                      if l.strip() and l.strip()[0].isdigit()]
        if data_lines:
            print(f'  Erste Zeile: {repr(data_lines[0][:65])}')
            print(f'  Letzte Zeile: {repr(data_lines[-1][:65])}')
        weekly_vals = []
        weekly_dates = []
        for line in data_lines:
            date_part = line[:10].strip()
            rest = line[10:]
            nums = re.findall(r"[+-]?\d+\.\d+", rest)
            if len(nums) >= 6:
                try:
                    ssta = float(nums[5])  # Nino3.4 SSTA (0-basiert nach Datum)
                    if -4 <= ssta <= 4:
                        weekly_vals.append(ssta)
                        weekly_dates.append(date_part)
                except: continue
        if weekly_vals:
            enso_now = round(weekly_vals[-1], 2)
            enso_source = 'cpc_weekly'
            last_date_str = weekly_dates[-1]
            print(f'Weekly Nino3.4 SSTA: {enso_now:+.2f} degC  (letzte Woche: {last_date_str}, gesamt: {len(weekly_vals)})')
            # Aktualitaetspruefung: Daten aelter als 90 Tage → als veraltet markieren
            try:
                m = re.match(r'(\d{2})([A-Z]{3})(\d{4})', last_date_str.strip())
                if m:
                    import datetime as _dt
                    mon_map = {'JAN':1,'FEB':2,'MAR':3,'APR':4,'MAY':5,'JUN':6,
                               'JUL':7,'AUG':8,'SEP':9,'OCT':10,'NOV':11,'DEC':12}
                    last_dt = _dt.date(int(m.group(3)), mon_map[m.group(2)], int(m.group(1)))
                    age_days = (datetime.date.today() - last_dt).days
                    if age_days > 90:
                        enso_source = 'cpc_weekly_stale'
                        print(f'  ⚠️  Daten {age_days} Tage alt – als stale markiert')
                    else:
                        print(f'  ✅ Daten {age_days} Tage alt – aktuell')
            except: pass
        else:
            print('  Weekly: keine gültigen Werte')
    except Exception as e:
        print(f'  Weekly parse Fehler: {e}')

# ONI (3-Monats-Mittel) aus CPC ASCII
# Format: SEAS YR TOTAL CLIM ANOM
if raw.get('enso_oni_raw'):
    try:
        oni_vals = []
        for line in raw['enso_oni_raw'].splitlines():
            parts = line.split()
            if len(parts) >= 5 and parts[0] not in ['SEAS', '']:
                try:
                    anom = float(parts[4])
                    if -4 <= anom <= 4:
                        oni_vals.append(anom)
                except: continue
        if oni_vals:
            enso_oni = round(oni_vals[-1], 2)
            print(f'ONI (3-Monats): {enso_oni:+.2f} degC  ({len(oni_vals)} Einträge)')
    except Exception as e:
        print(f'  ONI parse Fehler: {e}')

# ── ENSO 5-Klassen-Klassifikation ────────────────────────────
# Ref: ONI wenn vorhanden (offiziell), sonst weekly Nino3.4
_enso_ref = enso_oni if enso_oni is not None else enso_now

if _enso_ref is None:
    _enso_phase_class  = 'unknown'
    _enso_macro_phase  = 'unknown'
    _enso_direction    = 'unknown'
    _enso_event_risk   = 'unknown'
elif _enso_ref <= -0.5:
    _enso_phase_class  = 'la_nina'
    _enso_macro_phase  = 'la_nina'
    _enso_event_risk   = 'active_event'
elif _enso_ref <= -0.2:
    _enso_phase_class  = 'cool_neutral'
    _enso_macro_phase  = 'neutral'
    _enso_event_risk   = 'la_nina_possible'
elif _enso_ref <= +0.2:
    _enso_phase_class  = 'neutral'
    _enso_macro_phase  = 'neutral'
    _enso_event_risk   = 'neutral'
elif _enso_ref <= +0.5:
    _enso_phase_class  = 'warm_neutral'
    _enso_macro_phase  = 'neutral'
    _enso_event_risk   = 'el_nino_likely'
else:
    _enso_phase_class  = 'el_nino_candidate'
    _enso_macro_phase  = 'el_nino'
    _enso_event_risk   = 'active_event'

# ONI + weekly beide bestätigt → offiziell hochstufen
if enso_oni is not None and enso_now is not None:
    if enso_oni >= 0.5 and enso_now >= 0.5:
        _enso_phase_class = 'el_nino'
        _enso_macro_phase = 'el_nino'
        _enso_event_risk  = 'active_event'
    elif enso_oni <= -0.5 and enso_now <= -0.5:
        _enso_phase_class = 'la_nina'
        _enso_macro_phase = 'la_nina'
        _enso_event_risk  = 'active_event'

# Richtung aus weekly Trend (letzten 4 Wochen)
if len(weekly_vals) >= 4:
    recent_trend = weekly_vals[-1] - weekly_vals[-4]
    if recent_trend > 0.1:    _enso_direction = 'warming'
    elif recent_trend < -0.1: _enso_direction = 'cooling'
    else:                     _enso_direction = 'stable'
else:
    _enso_direction = 'unknown'

# Rückwärtskompatibilität
_enso_phase_obs  = _enso_phase_class
_enso_phase_fcst = 'not_used'

print(f'ENSO Klassifikation:')
print(f'  phase_class:  {_enso_phase_class}')
print(f'  macro_phase:  {_enso_macro_phase}')
print(f'  direction:    {_enso_direction}')
print(f'  event_risk:   {_enso_event_risk}')
print(f'  Nino3.4:      {enso_now} °C  (weekly)')
print(f'  ONI:          {enso_oni} °C  (3-Monats)')

---
## 3. Visualisierungen

In [ ]:
# ============================================================
# OBERFLÄCHENPARAMETER – Vergleich der 6 Messpunkte
# ============================================================

if not df_surf.empty:
    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=[
            'Temperatur 2m [°C]', 'Relative Feuchte [%]', 'Windgeschwindigkeit [km/h]',
            'Luftdruck [hPa]', 'Bodenfeuchtigkeit [m³/m³]', 'Niederschlag [mm]'
        ],
        vertical_spacing=0.18, horizontal_spacing=0.1
    )

    point_colors = ['#E85D24','#185FA5','#888780','#378ADD','#639922','#F2A623']
    params = [
        ('temp_2m', 1, 1), ('rh',       1, 2), ('wind',      1, 3),
        ('pressure',2, 1), ('soil_moist',2, 2), ('precip_day',2, 3),
    ]
    for col_name, row, col in params:
        vals = df_surf[col_name].fillna(0).tolist()
        fig.add_trace(go.Bar(
            x=df_surf['point'].tolist(), y=vals,
            marker_color=point_colors[:len(vals)], opacity=0.82,
            showlegend=False
        ), row=row, col=col)

    fig.update_layout(
        title=dict(text='Layer 2 – Oberflächenparameter (Open-Meteo, 6 Referenzpunkte)', font=dict(size=14)),
        height=500, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=50, r=30, t=60, b=80)
    )
    fig.update_xaxes(tickangle=30)
    fig.show()

In [ ]:
# ============================================================
# KONVEKTIONS-POTENTIAL PRO MESSPUNKT
# Konvektion = f(T, RH, Wind, Soil Moisture)
# ============================================================

def conv_potential(row):
    """Normierter Konvektions-Index [0,1] aus Oberflächenparametern."""
    t   = row.get('temp_2m')  or 0
    rh  = row.get('rh')       or 0
    w   = row.get('wind')     or 0
    sm  = row.get('soil_moist') or 0
    # Hohe Temp + hohe Feuchte + geringe Windscherung → hohe Konvektion
    score = (max(0, t - 10) / 40) * 0.4 + (rh / 100) * 0.35 + (sm * 10) * 0.15 + (1 - min(w, 50) / 50) * 0.1
    return round(min(1.0, max(0.0, score)), 3)

df_surf['conv_potential'] = df_surf.apply(lambda r: conv_potential(r.to_dict()), axis=1)

if not df_surf.empty:
    fig = go.Figure()
    colors_conv = ['#2ecc71' if v < 0.4 else '#f39c12' if v < 0.65 else '#e74c3c'
                   for v in df_surf['conv_potential']]

    fig.add_trace(go.Bar(
        x=df_surf['point'], y=df_surf['conv_potential'],
        marker_color=colors_conv, opacity=0.85,
        text=[f'{v:.2f}' for v in df_surf['conv_potential']],
        textposition='outside'
    ))
    fig.add_hline(y=0.65, line_dash='dot', line_color='#e74c3c',
                  annotation_text='Hohe Konvektion')
    fig.add_hline(y=0.40, line_dash='dot', line_color='#f39c12',
                  annotation_text='Moderate Konvektion')
    fig.update_layout(
        title=dict(text='Konvektions-Potential pro Messpunkt (Layer 2)', font=dict(size=14)),
        yaxis=dict(title='Konvektions-Index [0–1]', range=[0, 1.15]),
        height=380, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=60, r=30, t=55, b=80), showlegend=False
    )
    fig.show()

In [ ]:
# ============================================================
# ENSO + SST-ANOMALIE
# ============================================================

if raw["sst_anomaly"] and raw["sst_anomaly"].get("data"):
    vals = raw["sst_anomaly"]["data"]
    sorted_keys = sorted(vals.keys())[-60:]
    sst_vals = []
    for k in sorted_keys:
        try:
            raw_v = vals[k]
            # API liefert entweder float oder {'anomaly': float}
            v = float(raw_v['anomaly'] if isinstance(raw_v, dict) else raw_v)
            sst_vals.append(v if -5 < v < 5 else None)
        except:
            sst_vals.append(None)

    sst_clean = [v for v in sst_vals if v is not None]
    if not sst_clean:
        print("SST: keine gültigen Werte.")
    else:
        print(f"SST: min={min(sst_clean):.3f}  max={max(sst_clean):.3f}  aktuell={sst_clean[-1]:.3f} degC")
        y_margin = max(0.8, max(abs(v) for v in sst_clean) * 1.25)
        colors_sst = ["#e74c3c" if (v or 0) > 0.5
                      else "#378ADD" if (v or 0) < -0.5
                      else "#888780"
                      for v in sst_vals]
        phase_str = ""
        if enso_now is not None:
            phase = "El Nino" if enso_now > 0.5 else "La Nina" if enso_now < -0.5 else "neutral"
            phase_str = f" | ENSO: {phase} ({enso_now:+.2f} degC)"
        fig = go.Figure()
        fig.add_trace(go.Bar(x=sorted_keys, y=sst_vals,
                             marker_color=colors_sst, opacity=0.82, name="SST-Anomalie"))
        fig.add_hline(y= 0.5, line_dash="dot", line_color="#e74c3c", annotation_text="El Nino-Schwelle")
        fig.add_hline(y=-0.5, line_dash="dot", line_color="#378ADD", annotation_text="La Nina-Schwelle")
        fig.add_hline(y=0, line_color="gray", line_width=0.5)
        fig.update_layout(
            title=dict(text=f"Globale SST-Anomalie (Ozean, letzte 60 Monate){phase_str}", font=dict(size=14)),
            yaxis=dict(title="Anomalie [degC]", range=[-y_margin, y_margin]),
            height=330, plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)",
            margin=dict(l=60, r=30, t=55, b=50), showlegend=False
        )
        fig.show()
else:
    print("SST-Daten nicht verfügbar.")



In [ ]:
# ============================================================
# BODEN-ATMOSPHAERE-KOPPLUNG – Heatmap
# Zeigt welche Punkte welche Kopplungsstaerke haben
# ============================================================

if not df_surf.empty:
    def elec_coupling(row):
        """Elektrisches Kopplungs-Potential [0,1]: Boden-Atm. E-Feld-Austausch."""
        rh  = (row.get('rh')       or 50) / 100
        sm  = (row.get('soil_moist') or 0.1) * 5   # skaliert
        pre = min(1.0, (row.get('precip_day') or 0) / 20)
        # Hohe Feuchte + Regen → hohe Leitfähigkeit → starke E-Feld-Kopplung
        return round(min(1.0, rh * 0.4 + sm * 0.35 + pre * 0.25), 3)

    df_surf['elec_coupling'] = df_surf.apply(lambda r: elec_coupling(r.to_dict()), axis=1)

    metrics = ['conv_potential', 'elec_coupling']
    labels  = ['Konvektions-\nPotential', 'Elektrische\nKopplung']
    z_data  = [df_surf[m].tolist() for m in metrics]

    fig = go.Figure(go.Heatmap(
        z=z_data,
        x=df_surf['point'].tolist(),
        y=labels,
        colorscale='RdYlGn',
        zmin=0, zmax=1,
        text=[[f'{v:.2f}' for v in row] for row in z_data],
        texttemplate='%{text}',
        textfont=dict(size=12),
        colorbar=dict(title='Index [0-1]')
    ))
    fig.update_layout(
        title=dict(text='Boden-Atmosphäre-Kopplung: Konvektion & E-Feld (Layer 2)', font=dict(size=14)),
        height=280, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=130, r=80, t=55, b=80)
    )
    fig.show()

---
## 4. Zustandsbewertung & Übergabe an Layer 3

In [ ]:
# ============================================================
# LAYER-2-SCORE
# Fuenf dynamische Komponenten
# ============================================================

def norm(v, lo, hi):
    if v is None: return None
    return round(max(0.0, min(1.0, (v - lo) / (hi - lo))), 4)

# --- 1) Konvektions-Potential (global, Mittelwert der Punkte) ---
conv_score_val = None
conv_source    = 'missing'
if not df_surf.empty and 'conv_potential' in df_surf.columns:
    conv_score_val = round(float(df_surf['conv_potential'].mean()), 4)
    conv_source    = 'primary'

# --- 2) SST-Anomalie (Ozean-Energieinhalt) ---
sst_score_val  = None
if sst_anomaly_now is not None:
    # Anomalie -1 bis +2 degC → normiert: 0 = kuehler Ozean, 1 = sehr warm
    sst_score_val = norm(sst_anomaly_now, -1.0, 2.0)

# --- 3) ENSO-Phase (Grossraum-Modulator) ---
# Bevorzuge weekly Nino3.4, Fallback ONI
_enso_ref = enso_now if enso_now is not None else enso_oni
enso_score_val = None
enso_score_src = enso_source if enso_now is not None else ('cpc_oni' if enso_oni is not None else 'missing')
if _enso_ref is not None:
    enso_score_val = norm(_enso_ref, -2.0, 2.0)
    print(f'ENSO Score: {enso_score_val:.3f} (ref={_enso_ref:+.2f}, src={enso_score_src})')

# --- 4) Elektrische Kopplung (Mittelwert Punkte) ---
elec_score_val = None
elec_source    = 'missing'
if not df_surf.empty and 'elec_coupling' in df_surf.columns:
    elec_score_val = round(float(df_surf['elec_coupling'].mean()), 4)
    elec_source    = 'primary'

# --- 5) Tropenfeuchte (Bodenfeuchtigkeit, Proxy fuer globalen Wasserkreislauf) ---
moist_score_val = None
moist_source    = 'missing'
if not df_surf.empty and 'soil_moist' in df_surf.columns:
    sm_mean = df_surf['soil_moist'].dropna().mean()
    if not pd.isna(sm_mean):
        moist_score_val = norm(float(sm_mean), 0.0, 0.5)
        moist_source = 'primary'

# --- Zusammenfassen ---
COMPONENTS = {
    'Konvektions-Potential':  {'score': conv_score_val,  'source': conv_source,    'dynamic': True},
    'SST-Anomalie (Ozean)':   {'score': sst_score_val,   'source': sst_source,     'dynamic': True},
    'ENSO-Phase (Nino3.4)':   {'score': enso_score_val,  'source': enso_score_src, 'dynamic': True},
    'Elektr. Bodenkopplung':  {'score': elec_score_val,  'source': elec_source,    'dynamic': True},
    'Bodenfeuchte (global)':  {'score': moist_score_val, 'source': moist_source,   'dynamic': True},
}

available    = {k: v['score'] for k, v in COMPONENTS.items() if v['score'] is not None}
unavailable  = [k for k, v in COMPONENTS.items() if v['score'] is None]
layer2_score = round(sum(available.values()) / len(available), 4) if available else None
confidence   = round(len(available) / len(COMPONENTS), 2)
level = ('unbekannt' if layer2_score is None
         else 'ruhig'   if layer2_score < 0.3
         else 'moderat' if layer2_score < 0.6
         else 'aktiv')

# Dominant Driver
if available:
    dominant_l2 = max(available, key=available.get)
else:
    dominant_l2 = 'none'

# --- Ausgabe ---
W = 66
print('=' * W)
print('LAYER 2 – OBERFLÄCHE & KONTAKTZONE – ZUSTANDSBEWERTUNG')
print('=' * W)
for name, comp in COMPONENTS.items():
    s = comp['score']
    if s is not None:
        bar = '█' * int(s * 20) + '░' * (20 - int(s * 20))
        print(f'  {name:<32} {bar}  {s:.3f}  [{comp["source"]}]')
    else:
        print(f'  {name:<32} {"─" * 20}  n/a   [missing]')
print('-' * W)
print(f'  Score:          {layer2_score:.3f}  ({len(available)}/{len(COMPONENTS)} Komponenten)')
print(f'  Confidence:     {confidence:.0%}')
print(f'  Level:          {level.upper()}')
print(f'  Dominant:       {dominant_l2}')
if layer1:
    print(f'  L1-Eingang:     {layer1["level"].upper()} | Seismik-elevated: {layer1.get("flags",{}).get("elevated_seismicity","–")}')
print('=' * W)

# Radar
cats = list(available.keys())
vals_r = list(available.values())
if len(cats) >= 3:
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=vals_r + [vals_r[0]], theta=cats + [cats[0]],
        fill='toself', fillcolor='rgba(99,153,34,0.20)',
        line=dict(color='#639922', width=2.5), name='Layer 2'
    ))
    fig.add_trace(go.Scatterpolar(
        r=[0.5] * (len(cats)+1), theta=cats + [cats[0]],
        line=dict(color='#E24B4A', dash='dot', width=1),
        mode='lines', name='Aktivitätsschwelle'
    ))
    fig.update_layout(
        title=dict(
            text=f'Layer 2 – Aktivitätsprofil | Score: {layer2_score:.3f} | {level.upper()} | Confidence: {confidence:.0%}',
            font=dict(size=13)
        ),
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        height=440, showlegend=True, margin=dict(l=60, r=60, t=65, b=40)
    )
    fig.show()


In [ ]:
# ============================================================
# EXPORT – layer2_state.json
# ============================================================

# ENSO-Phase
# Phasen-Logik: Weekly-Wert allein reicht nicht fuer 'El Nino' – ONI benoetigt
_enso_ref = enso_now if enso_now is not None else enso_oni
if _enso_ref is None:
    _enso_phase_obs  = 'unknown'
    _enso_phase_fcst = 'not_available'
elif _enso_ref >= 1.5:
    _enso_phase_obs  = 'strong_el_nino' if enso_oni is not None else 'warm_neutral'
    _enso_phase_fcst = 'el_nino_active'
elif _enso_ref >= 0.5:
    _enso_phase_obs  = 'warm_neutral'
    _enso_phase_fcst = 'el_nino_likely'
elif _enso_ref >= 0.0:
    _enso_phase_obs  = 'neutral'
    _enso_phase_fcst = 'neutral_to_warm'
elif _enso_ref >= -0.5:
    _enso_phase_obs  = 'neutral'
    _enso_phase_fcst = 'neutral_to_cool'
elif _enso_ref >= -1.5:
    _enso_phase_obs  = 'cool_neutral'
    _enso_phase_fcst = 'la_nina_likely'
else:
    _enso_phase_obs  = 'strong_la_nina' if enso_oni is not None else 'cool_neutral'
    _enso_phase_fcst = 'la_nina_active'
_enso_phase = _enso_phase_obs

# Schumann-Downstream (aus Konvektion und elektr. Kopplung)
_schumann = (
    'erhoehte Gewitteraktivitaet moeglich – Schumann-Anregung erwartet'
    if conv_score_val is not None and conv_score_val > 0.65
    else 'moderate Konvektion – leichte Schumann-Modulation moeglich'
    if conv_score_val is not None and conv_score_val > 0.40
    else 'geringe Konvektion – kein starker Schumann-Treiber aus Layer 2'
)

# Gewitterpotential (Trigger-Flag fuer Layer 3)
_thunder_flag = (
    conv_score_val is not None and conv_score_val > 0.55
    and elec_score_val is not None and elec_score_val > 0.45
)

# state_summary (nur ASCII fuer JSON-Sicherheit)
_sst_str   = (f'SST-Anomalie {sst_anomaly_now:+.2f} degC.' if sst_anomaly_now is not None
              else 'SST nicht verfuegbar.')
_enso_ref_val = enso_now if enso_now is not None else enso_oni
_w = f'{enso_now:+.2f}' if enso_now is not None else 'n/a'
_o = f'{enso_oni:+.2f}' if enso_oni is not None else 'n/a'
_enso_str = (f'ENSO: {_enso_phase_obs} / {_enso_phase_fcst} (weekly Nino3.4={_w}, ONI={_o}; Forecast nicht im Score).' if _enso_ref_val is not None
             else 'ENSO: Daten nicht verfuegbar.')
_conv_str  = (f'Konvektions-Potential: {conv_score_val:.2f}.' if conv_score_val is not None
              else 'Konvektion: nicht verfuegbar.')
_l1_str    = (f'L1-Eingang: {layer1["level"]} (Score {layer1["score"]:.3f}).' if layer1
              else 'L1-Kontext fehlt.')

state_summary = ' '.join([
    f'Layer-2-Zustand: {level}.', _sst_str, _enso_str, _conv_str,
    f'Datenvollstaendigkeit: {confidence:.0%}.', _l1_str
])

layer2_state = {
    'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
    'layer': 2,
    'name':  'Oberfläche & Biosphärennahe Kontaktzone',

    # Score
    'score':      layer2_score,
    'level':      level,
    'confidence': confidence,
    'score_basis': f'{len(available)}/{len(COMPONENTS)} dynamische Komponenten',
    'dominant_component': dominant_l2,
    'missing_components': unavailable,

    # Komponenten
    'components': {
        k: {
            'score':   round(v['score'], 4) if v['score'] is not None else None,
            'source':  v['source'],
            'dynamic': v['dynamic'],
        }
        for k, v in COMPONENTS.items()
    },

    # Rohdaten
    'raw_values': {
        'SST_anomaly_degC':  round(sst_anomaly_now, 3) if sst_anomaly_now is not None else None,
        'ENSO': {
            'weekly_nino34_anomaly_degC': enso_now,
            'oni_3month_degC':            enso_oni,
            'phase_class':                _enso_phase_class,    # neu: 5-Klassen
            'macro_phase':                _enso_macro_phase,    # neu
            'direction':                  _enso_direction,      # neu
            'event_risk':                 _enso_event_risk,     # neu
            'phase_observed':             _enso_phase_obs,      # bleibt für Rückwärtskompatibilität
            'phase_forecast':             _enso_phase_fcst,
            'note':                       'phase_forecast nicht im Score verwendet',
            'source':                     enso_source,
                },
        'conv_potential_mean': round(conv_score_val, 3) if conv_score_val is not None else None,
        'elec_coupling_mean':  round(elec_score_val, 3) if elec_score_val is not None else None,
        'surface_points': [
            {
                'point':    row['point'],
                'temp_2m':  row.get('temp_2m'),
                'rh':       row.get('rh'),
                'conv':     row.get('conv_potential'),
                'elec':     row.get('elec_coupling'),
            }
            for row in df_surf.to_dict('records')
        ],
        },
        #Flags
        'flags': {
        'elevated_convection':  (conv_score_val > 0.55) if conv_score_val is not None else None,
        'el_nino_active':       _enso_phase_class in ('el_nino',),
        'el_nino_developing':   _enso_phase_class in ('warm_neutral', 'el_nino_candidate'),
        'la_nina_active':       _enso_phase_class in ('la_nina',),
        'la_nina_developing':   _enso_phase_class in ('cool_neutral',),   # neu
        'sst_anomaly_high':     (sst_anomaly_now > 0.5) if sst_anomaly_now is not None else None,
        'thunderstorm_trigger': bool(_thunder_flag),
        'bz_orientation':       layer0.get('flags', {}).get('bz_orientation') if layer0 else None,
        },

    # Downstream-Erwartung
    'downstream_expectation': {
        'layer3_weather':    ('erhoehtes Gewitterpotential – Konvektion und E-Kopplung aktiv'
                              if _thunder_flag else
                              'moderate Konvektion – normale Gewitteraktivitaet'
                              if conv_score_val is not None and conv_score_val > 0.35 else
                              'niedrige Konvektion – wenig Wetterantrieb aus Layer 2'),
        'layer5_gec':        ('erhoehter Energieeintrag in GEC durch Gewitteraktivitaet moeglich'
                              if _thunder_flag else
                              'normaler GEC-Grundzustand erwartet'),
        'schumann_resonance': _schumann,
        'layer4_ionosphere': ('ENSO-modulierte Konvektion beeinflusst troposphaerische Wellenausbreitung'
                              if _enso_phase != 'neutral' else
                              'kein starker ENSO-Treiber auf Ionosphaere'),
    },

    # Kontext aus vorherigen Layern
    'layer0_context': {
        'score': layer0['score'] if layer0 else None,
        'level': layer0['level'] if layer0 else None,
        'dominant_driver': layer0.get('dominant_driver') if layer0 else None,
    },
    'layer1_context': {
        'score':  layer1['score']  if layer1 else None,
        'level':  layer1['level']  if layer1 else None,
        'seismic_elevated': layer1.get('flags', {}).get('elevated_seismicity') if layer1 else None,
    },

    'state_summary': state_summary,
}

with open(layer_state(2), 'w', encoding='utf-8') as f:
    json.dump(layer2_state, f, indent=2, ensure_ascii=False)

print(f'gespeichert: {layer_state(2)}')
print(json.dumps(layer2_state, indent=2, ensure_ascii=False))



---
## Zusammenfassung Layer 2

| Aspekt | Inhalt |
|--------|--------|
| **Rolle** | Grenzschicht Erdoberfläche–Atmosphäre, Kopplungszone |
| **Datenquellen** | Open-Meteo (6 Punkte), NOAA SST, NOAA ENSO Nino3.4 |
| **Dynamische Komponenten** | Konvektion, SST-Anomalie, ENSO, elektr. Kopplung, Bodenfeuchte |
| **→ Layer 3** | Konvektions-Potential → Gewitterbildung, Wettergeschehen |
| **→ Layer 5** | Gewittertrigger → Global Electric Circuit |
| **→ Layer 6** | Konvektion + E-Kopplung → Schumann-Anregung |
| **Ausgabe** | `layer2_state.json` mit L0/L1-Kontext und thunderstorm_trigger |

> **Nächster Schritt:** `atmosphere_analysis_layer3.ipynb` – Atmosphäre / Wetter / Gewitter